In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments
)
from transformers.trainer_utils import get_last_checkpoint

In [ ]:
# 1. Mount Google Drive (Colab specific)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Use Drive for saving checkpoints
    base_dir = "/content/drive/MyDrive/phinc/mbart"
    output_dir = os.path.join(base_dir, "checkpoints") # Save checkpoints in a 'checkpoints' subdirectory
    final_output_dir = os.path.join(base_dir, "final_model") # Define final model directory within base_dir
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Using local directories.")
    output_dir = os.path.join("./mbart-phinc", "checkpoints")
    final_output_dir = os.path.join("./mbart-phinc", "final_model")

# Ensure directories exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(final_output_dir, exist_ok=True)

In [ ]:
# 2. Load Dataset
print("Loading dataset...")
dataset = load_dataset("LingoIITGN/PHINC")
# Split the dataset
raw_split = dataset["train"].train_test_split(test_size=0.1, seed=42)
dataset = raw_split
print(f"Dataset loaded: {dataset}")

In [ ]:
# 3. Initialize Model and Tokenizer
model_name = "facebook/mbart-large-50-many-to-many-mmt"
print(f"Loading model: {model_name}")
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

In [ ]:
# 4. Preprocess Data
max_length = 128

def preprocess(batch):
    inputs = batch["Sentence"]              # Hinglish
    targets = batch["English_Translation"]  # English

    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_data = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)

### 4.1 Save Tokenized Data Splits to Google Drive (Optional but Recommended)

To ensure consistency and to avoid re-tokenizing the dataset in future sessions, we'll save the tokenized train and test splits to Google Drive. This also allows for easy resumption or sharing of the exact data used for training.

In [ ]:
# Define directories for saving tokenized data
tokenized_data_dir = os.path.join(base_dir, "tokenized_data")
os.makedirs(tokenized_data_dir, exist_ok=True)

# Save train split
train_output_path = os.path.join(tokenized_data_dir, "train")
print(f"Saving tokenized train data to {train_output_path}...")
tokenized_data["train"].save_to_disk(train_output_path)

# Save test split
test_output_path = os.path.join(tokenized_data_dir, "test")
print(f"Saving tokenized test data to {test_output_path}...")
tokenized_data["test"].save_to_disk(test_output_path)

print("Tokenized data splits saved successfully.")


In [ ]:
from datasets import load_from_disk
import os

# Ensure tokenized_data_dir is defined (it should be from klkFsd_lV_sG)
# If running this cell independently, ensure tokenized_data_dir is set, e.g.:
# base_dir = "/content/drive/MyDrive/phinc/mbart" # or your local path
# tokenized_data_dir = os.path.join(base_dir, "tokenized_data")

if 'tokenized_data_dir' not in locals():
    print("tokenized_data_dir is not defined. Please run the setup cells first.")
else:
    train_path = os.path.join(tokenized_data_dir, "train")
    test_path = os.path.join(tokenized_data_dir, "test")

    if os.path.exists(train_path) and os.path.exists(test_path):
        print(f"Loading tokenized data from {tokenized_data_dir}...")
        tokenized_data = {
            "train": load_from_disk(train_path),
            "test": load_from_disk(test_path)
        }
        print("Tokenized data loaded successfully.")
    else:
        print("No saved tokenized data found. Please run the data tokenization and saving cells first.")


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

In [ ]:
# 5. Training Arguments
print(f"Checkpoints will be saved to: {output_dir}")
training_args = TrainingArguments(
    output_dir=output_dir,          # Save checkpoints directly to Drive (or local if not Colab)
    eval_strategy="epoch",          # Evaluate every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,                      # Use mixed precision
    logging_steps=100,
    save_total_limit=2,             # Keep only the last 2 checkpoints to manage space
    save_strategy="epoch",          # Save checkpoint every epoch
    report_to="none",
    load_best_model_at_end=True,    # Load the best model at the end of training
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    data_collator=data_collator
)

In [ ]:
# 6. Train with Resume Capability
# Check if a checkpoint exists in the output directory
try:
    last_checkpoint = get_last_checkpoint(output_dir)
except Exception:
    last_checkpoint = None

if last_checkpoint:
    print(f"Found checkpoint: {last_checkpoint}. Resuming training.")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("No checkpoint found. Starting training from scratch.")
    trainer.train()

In [ ]:
# 7. Save Final Model
print(f"Saving final model to {final_output_dir}...")
trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)
print("Training complete and model saved.")

In [ ]:
from datasets import load_dataset

# Load the COMI-LINGUA dataset
dataset = load_dataset("LingoIITGN/PHINC")
print(dataset)

# Print a few examples from the 'test' split to show actual data
print("\n--- Sample Data from 'test' split ---")
for i in range(10):
    print(f"Source ({i+1}): {dataset['train'][i]['Sentence']}")
    print(f"Target ({i+1}): {dataset['train'][i]['English_Translation']}")
    print("-------------------------------------")

In [ ]:
len(dataset['train'])